# Migrando os jsons para o banco de Dados


## Start

In [ ]:
import json
import psycopg
import os
from dotenv import load_dotenv

load_dotenv()
DB_URL = os.getenv("DATABASE_URL")

def get_conn():
    return psycopg.connect(DB_URL)

print("Setup concluído. Pronto para migrar!")

## Agentes e Mapas

In [ ]:
with get_conn() as conn:
    with conn.cursor() as cur:
        # Migrar Agentes
        with open('./botAPI/VLR_agents.json', 'r', encoding='utf-8') as f:
            agentes = json.load(f)
            for a in agentes:
                cur.execute("INSERT INTO agentes (id, nome, emoji_discord) VALUES (%s, %s, %s) ON CONFLICT (id) DO NOTHING", 
                            (a['id'], a['nome'], a.get('img')))
        
        # Migrar Mapas
        with open('./botAPI/VLR_mapas.json', 'r', encoding='utf-8') as f:
            mapas = json.load(f)
            for m in mapas:
                cur.execute("INSERT INTO mapas_lista (id, nome) VALUES (%s, %s) ON CONFLICT (id) DO NOTHING", 
                            (m['id'], m['nome']))
    conn.commit()
print("Agentes e Mapas migrados!")

## Times

In [ ]:
with open('./botAPI/times.json', 'r', encoding='utf-8') as f:
    times = json.load(f)

with get_conn() as conn:
    with conn.cursor() as cur:
        for t in times:
            cur.execute("""
                INSERT INTO times (id, nome, tag, regiao, emoji, img_url)
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT (id) DO UPDATE SET nome = EXCLUDED.nome
            """, (t['TIME_ID'], t['nome'], t['tag'], t['regiao'], t['emoji'], t['img']))
    conn.commit()
print(f"{len(times)} times migrados!")

## Players


In [ ]:
with open('./botAPI/players.json', 'r', encoding='utf-8') as f:
    players = json.load(f)

with get_conn() as conn:
    with conn.cursor() as cur:
        for p in players:
            cur.execute("""
                INSERT INTO players (id, nome, time_id)
                VALUES (%s, %s, %s)
                ON CONFLICT (id) DO UPDATE SET time_id = EXCLUDED.time_id
            """, (p['PLAYER_ID'], p['nome'], p['time']))
    conn.commit()
print(f"{len(players)} players migrados!")

## Partidas, Composições e Mapas_Jogados

In [ ]:
def migrar_partidas_v2():
    # Caminho conforme sua organização de pastas
    path = './botAPI/partidas.json'
    
    if not os.path.exists(path):
        print(f"❌ Arquivo não encontrado em: {path}")
        return

    with open(path, 'r', encoding='utf-8') as f:
        partidas = json.load(f)

    try:
        with get_conn() as conn:
            with conn.cursor() as cur:
                print(f"Iniciando processamento de {len(partidas)} partidas...")

                for p in partidas:
                    # Mapeamento: timeA/B é uma lista [ID_A, ID_B]
                    id_partida = p['id']
                    id_time_a = p['timeA/B'][0]
                    id_time_b = p['timeA/B'][1]
                    
                    # Convertendo o dicionário de pickban para string JSON
                    pickban_str = json.dumps(p['pickban'])

                    # 1. Inserir/Atualizar a Partida
                    cur.execute("""
                        INSERT INTO partidas (id, camp_nome, timeA_id, timeB_id, pickban_log, vencedor_time_letra)
                        VALUES (%s, %s, %s, %s, %s, %s)
                        ON CONFLICT (id) DO UPDATE SET 
                            camp_nome = EXCLUDED.camp_nome,
                            vencedor_time_letra = EXCLUDED.vencedor_time_letra;
                    """, (id_partida, p['camp'], id_time_a, id_time_b, pickban_str, p['win']))

                    # 2. Processar a lista de mapas (MD3 ou MD5)
                    for mapa in p['mapas']:
                        
                        # Função interna para gerenciar as composições únicas
                        def get_or_create_comp(agentes_ids):
                            # Você confirmou que os IDs já vêm ordenados no seu JSON!
                            cur.execute("""
                                SELECT id FROM composicoes 
                                WHERE agente1=%s AND agente2=%s AND agente3=%s AND agente4=%s AND agente5=%s
                            """, agentes_ids)
                            res = cur.fetchone()
                            
                            if res:
                                return res[0]
                            else:
                                cur.execute("""
                                    INSERT INTO composicoes (agente1, agente2, agente3, agente4, agente5)
                                    VALUES (%s, %s, %s, %s, %s) RETURNING id
                                """, agentes_ids)
                                return cur.fetchone()[0]

                        # No seu JSON: composicoes[0] é o Time A, composicoes[1] é o Time B
                        id_comp_a = get_or_create_comp(mapa['composicoes'][0])
                        id_comp_b = get_or_create_comp(mapa['composicoes'][1])

                        # 3. Inserir o Mapa Jogador
                        # Note: 'atk' no seu JSON vira 'atk_start' no banco
                        cur.execute("""
                            INSERT INTO mapas_jogados (partida_id, mapa_id, atk_start, compA_id, compB_id, rounds_string, vencedor_mapa)
                            VALUES (%s, %s, %s, %s, %s, %s, %s)
                        """, (id_partida, mapa['id'], mapa['atk'], id_comp_a, id_comp_b, mapa['rounds'], mapa['win']))

                conn.commit()
                print("🚀 Migração concluída com sucesso!")

    except Exception as e:
        print(f"❌ Erro durante a migração: {e}")

# Executa a função
migrar_partidas_v2()

## Validação

In [ ]:
def validar_migracao():
    try:
        with get_conn() as conn:
            with conn.cursor() as cur:
                print("--- 📊 RELATÓRIO DE VALIDAÇÃO ---")
                
                # 1. Contagem Geral
                tabelas = ['times', 'players', 'agentes', 'mapas_lista', 'partidas', 'mapas_jogados', 'composicoes']
                for tabela in tabelas:
                    cur.execute(f"SELECT COUNT(*) FROM {tabela}")
                    count = cur.fetchone()[0]
                    print(f"✅ {tabela.capitalize()}: {count} registros found.")

                print("\n--- 🔍 TESTE DE RELACIONAMENTO (JOIN) ---")
                
                # 2. Teste de Integridade: Ver as últimas 3 partidas com nomes dos times
                cur.execute("""
                    SELECT p.id, tA.nome, tB.nome, p.vencedor_time_letra 
                    FROM partidas p
                    JOIN times tA ON p.timeA_id = tA.id
                    JOIN times tB ON p.timeB_id = tB.id
                    ORDER BY p.id DESC
                    LIMIT 3
                """)
                rows = cur.fetchall()
                for r in rows:
                    print(f"Partida ID {r[0]}: {r[1]} vs {r[2]} | Vencedor: {r[3]}")

                # 3. Teste de Composição: Ver um mapa aleatório e os agentes usados
                print("\n--- 🧙 TESTE DE COMPOSIÇÃO (AGENTES) ---")
                cur.execute("""
                    SELECT mj.id, m.nome, c.agente1, c.agente2, c.agente3, c.agente4, c.agente5
                    FROM mapas_jogados mj
                    JOIN mapas_lista m ON mj.mapa_id = m.id
                    JOIN composicoes c ON mj.compA_id = c.id
                    LIMIT 1
                """)
                m = cur.fetchone()
                if m:
                    print(f"Mapa: {m[1]} | IDs Agentes Time A: {list(m[2:])}")

    except Exception as e:
        print(f"❌ Erro na validação: {e}")

# Executa a validação
validar_migracao()

## Encerrar

In [ ]:
try:
    # Se você criou uma variável 'conn' global em alguma célula:
    if 'conn' in locals() and conn:
        conn.close()
        print("🔒 Conexão fechada manualmente com sucesso.")
    else:
        print("✅ Nenhuma conexão aberta detectada para fechar.")
except Exception as e:
    print(f"⚠️ Erro ao fechar conexão: {e}")